# Healthcare QA Chatbot — Google Colab Evaluation Runner

**Runtime:** Set to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`) before running.

## What this notebook does (in order):
| Cell | Task | IMPROVEMENTS.md ref |
|------|------|---------------------|
| 1 | Mount Drive + Clone Repo | Setup |
| 2 | Install Dependencies | Setup |
| 3 | Copy Models & Knowledge Base from Drive | Setup |
| 4 | Environment + GPU Check | Setup |
| 5 | Full 97-Question Evaluation — TinyLlama | §2.1 |
| 6 | Ablation Study — 97 Questions | §2.4 |
| 7 | BioMistral 7B Comparative Evaluation | §4.1 |
| 8 | QLoRA Re-Training (300 steps) | §4.2 |
| 9 | QLoRA Model Evaluation | §4.2 |
| 10 | Baseline Comparisons (No-RAG, Dense-only, No-XAI) | §4.5 |
| 11 | Generate Paper-Quality Figures | §4.3 |
| 12 | Save Results to Drive / GitHub | Wrap-up |

## Before you start:
1. Upload these two folders to your Google Drive under `MyDrive/healthcare_qa_data/`:
   - `models/` (contains `biomistral/ggml-model-Q4_K_M.gguf` and `fine_tuned/`)
   - `data/knowledge_base/` (ChromaDB, 2.9 GB)
2. Push your project to a **private GitHub repo** and set `GITHUB_REPO` below.
3. Run cells **top to bottom**. Each cell prints its status.

In [ ]:
# ============================================================
# CONFIGURATION — edit these values before running
# ============================================================
GITHUB_REPO = "https://github.com/kbssrikar7/healthcare-qa-chatbot.git"
DRIVE_BASE = "/content/drive/MyDrive/healthcare_qa_data"  # folder in your Drive
PROJECT_DIR = "/content/project"
GITHUB_TOKEN = ""  # optional: personal access token for pushing results back

print("Configuration set.")
print(f"  Repo      : {GITHUB_REPO}")
print(f"  Drive base: {DRIVE_BASE}")
print(f"  Project   : {PROJECT_DIR}")

## Cell 1 — Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
import os
import subprocess
import shutil

# Mount Drive
drive.mount("/content/drive")
print("Drive mounted.")

# Clone repo (or pull if already cloned)
if os.path.exists(PROJECT_DIR):
    print("Project dir exists — pulling latest changes...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)
else:
    print(f"Cloning {GITHUB_REPO} ...")
    subprocess.run(["git", "clone", GITHUB_REPO, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print("Repo contents:", os.listdir(PROJECT_DIR))

## Cell 2 — Install Dependencies
_This takes ~4-6 minutes on a fresh Colab instance._

In [ ]:
import subprocess
import sys


def run(cmd, **kw):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
    else:
        print("OK")
    return result


# 1. Upgrade pip quietly
run("pip install -q --upgrade pip")

# 2. Install PyTorch with CUDA 11.8 (Colab T4 default)
run(
    "pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118"
)

# 3. Install project requirements (minus unused extras)
#    We exclude lime, captum, wandb (unused) and install key packages explicitly
run(
    "pip install -q transformers>=4.36.0 peft>=0.7.0 accelerate>=0.25.0 safetensors bitsandbytes"
)
run("pip install -q sentence-transformers>=2.2.0 sentencepiece")
run("pip install -q langchain langchain-core langchain-community")
run("pip install -q chromadb>=0.4.0 faiss-cpu rank-bm25")
run("pip install -q spacy && python -m spacy download en_core_web_sm -q")
run("pip install -q evaluate rouge-score bert-score scikit-learn datasets")
run("pip install -q fastapi uvicorn python-multipart")
run("pip install -q pandas pyarrow tqdm loguru python-dotenv pyyaml psutil")
run("pip install -q shap")
run("pip install -q matplotlib seaborn")

# 4. llama-cpp-python with CUDA support (for BioMistral GGUF)
run(
    "CMAKE_ARGS='-DLLAMA_CUBLAS=on' pip install -q llama-cpp-python --force-reinstall --no-cache-dir"
)

print("\n✓ All dependencies installed.")

## Cell 3 — Copy Models & Knowledge Base from Google Drive
_Skip or adapt if you prefer to download models from HuggingFace directly._

In [ ]:
import os


def copy_from_drive(src, dst, label):
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.exists(dst):
            print(f"  {label}: already exists at {dst}, skipping copy.")
        else:
            print(f"  {label}: copying {src} → {dst} ...")
            if os.path.isdir(src):
                shutil.copytree(src, dst)
            else:
                shutil.copy2(src, dst)
            print(f"  {label}: done.")
    else:
        print(f"  {label}: NOT FOUND at {src} — will be downloaded/built later.")


# Knowledge base (ChromaDB, 2.9 GB)
copy_from_drive(
    f"{DRIVE_BASE}/knowledge_base",
    f"{PROJECT_DIR}/data/knowledge_base",
    "Knowledge Base (ChromaDB)",
)

# BioMistral GGUF model (4.1 GB)
copy_from_drive(
    f"{DRIVE_BASE}/models/biomistral",
    f"{PROJECT_DIR}/models/biomistral",
    "BioMistral GGUF",
)

# QLoRA adapter (49 MB) — optional, we will re-train anyway
copy_from_drive(
    f"{DRIVE_BASE}/models/fine_tuned",
    f"{PROJECT_DIR}/models/fine_tuned",
    "QLoRA adapter",
)

print("\nData copy step complete.")

## Cell 4 — Environment Setup & GPU Verification

In [ ]:
import os

# Add project to Python path
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# Environment variables
os.environ["USE_GPU"] = "true"
os.environ["HF_HUB_OFFLINE"] = "0"  # Allow HF downloads on Colab
os.environ["TRANSFORMERS_OFFLINE"] = "0"
os.environ["ENABLE_MCP_SEARCH"] = "false"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# GPU check
import torch

gpu_ok = torch.cuda.is_available()
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {gpu_ok}")
if gpu_ok:
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM             : {vram:.1f} GB")
else:
    print("WARNING: No GPU detected. Switch runtime to T4 GPU for best performance.")

# Show disk space

total, used, free = shutil.disk_usage("/")
print(f"\nDisk space       : {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total")

## Cell 5 — Full 97-Question Evaluation: TinyLlama  (§2.1)
Runs ROUGE-L, BERTScore, keyword coverage, latency, calibration on all 97 test cases.
Expected time: **~15-25 min** on T4 GPU.

In [ ]:
import sys
import os
import json
import time
import statistics
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# ── Load test set ────────────────────────────────────────────────────────────
TEST_SET_PATH = Path(PROJECT_DIR) / "evaluation" / "test_set_v2.json"
with open(TEST_SET_PATH) as f:
    all_cases = json.load(f)["test_cases"]
print(f"Loaded {len(all_cases)} test cases.")

# ── Load pipeline ────────────────────────────────────────────────────────────
print("Loading TinyLlama pipeline (this takes ~2-4 min on first run) ...")
from api.main import get_pipeline

pipeline = get_pipeline("tinyllama")
print("Pipeline loaded.")


# ── Metric helpers ───────────────────────────────────────────────────────────
def keyword_coverage(answer, keywords):
    if not keywords:
        return 1.0
    a = answer.lower()
    return sum(1 for k in keywords if k.lower() in a) / len(keywords)


# ── Run 97-question evaluation ───────────────────────────────────────────────
from rouge_score import rouge_scorer as _rs

scorer_rouge = _rs.RougeScorer(["rougeL"], use_stemmer=True)

preds, refs = [], []
kw_scores, rouge_scores, latencies = [], [], []
answerable = 0

print(f"\nRunning {len(all_cases)} questions ...")
for i, case in enumerate(all_cases, 1):
    try:
        t0 = time.perf_counter()
        resp = pipeline.answer(case["query"], include_explanation=False)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        latencies.append(elapsed_ms)

        answer = resp.answer if hasattr(resp, "answer") else str(resp)
        ref = case.get("reference_answer", "")
        kws = case.get("expected_keywords", [])

        kw_scores.append(keyword_coverage(answer, kws))
        if ref:
            rouge_scores.append(scorer_rouge.score(ref, answer)["rougeL"].fmeasure)
            preds.append(answer)
            refs.append(ref)
        if getattr(resp, "is_answerable", True):
            answerable += 1

        if i % 10 == 0 or i == len(all_cases):
            print(
                f"  {i}/{len(all_cases)}  kw_cov={statistics.mean(kw_scores):.3f}  "
                f"rouge={statistics.mean(rouge_scores) if rouge_scores else 0:.3f}  "
                f"lat={statistics.mean(latencies):.0f}ms"
            )
    except Exception as e:
        print(f"  [{i}] ERROR: {e}")

# ── BERTScore ────────────────────────────────────────────────────────────────
bs_f1_mean = 0.0
if preds:
    print("\nComputing BERTScore (batch) ...")
    import evaluate as ev

    bertscore_metric = ev.load("bertscore")
    try:
        bs = bertscore_metric.compute(
            predictions=preds,
            references=refs,
            lang="en",
            rescale_with_baseline=True,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
    except Exception as e:
        print(f"  BERTScore baseline not cached, using unscaled: {e}")
        bs = bertscore_metric.compute(
            predictions=preds,
            references=refs,
            lang="en",
            rescale_with_baseline=False,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
    bs_f1_mean = statistics.mean(bs["f1"])
    print(f"  BERTScore F1 mean: {bs_f1_mean:.4f}")

# ── Save results ─────────────────────────────────────────────────────────────
import numpy as np


def bootstrap_ci(scores, n_bootstrap=1000, ci=0.95):
    arr = np.array(scores)
    boot = [
        np.mean(np.random.choice(arr, size=len(arr), replace=True))
        for _ in range(n_bootstrap)
    ]
    return (
        float(np.mean(arr)),
        float(np.percentile(boot, (1 - ci) / 2 * 100)),
        float(np.percentile(boot, (1 + ci) / 2 * 100)),
    )


kw_mean, kw_lo, kw_hi = bootstrap_ci(kw_scores)
rouge_mean, r_lo, r_hi = bootstrap_ci(rouge_scores) if rouge_scores else (0, 0, 0)
bs_mean_v, bs_lo, bs_hi = bootstrap_ci(bs["f1"]) if preds else (0, 0, 0)

metrics_result = {
    "variant": "standard_tinyllama",
    "n": len(kw_scores),
    "answerable_pct": round(answerable / len(kw_scores), 4) if kw_scores else 0,
    "keyword_coverage_mean": round(kw_mean, 4),
    "keyword_coverage_ci_95": [round(kw_lo, 4), round(kw_hi, 4)],
    "rougeL_mean": round(rouge_mean, 4),
    "rougeL_ci_95": [round(r_lo, 4), round(r_hi, 4)],
    "bertscore_f1_mean": round(bs_f1_mean, 4),
    "bertscore_ci_95": [round(bs_lo, 4), round(bs_hi, 4)],
    "latency_mean_ms": round(statistics.mean(latencies), 1) if latencies else 0,
    "latency_p90_ms": round(sorted(latencies)[int(0.9 * len(latencies))], 1)
    if latencies
    else 0,
}

OUT_DIR = Path(PROJECT_DIR) / "evaluation" / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUT_DIR / "metrics_full_tinyllama.json", "w") as f:
    json.dump(metrics_result, f, indent=2)

print("\n" + "=" * 60)
print("TinyLlama 97-Question Results:")
for k, v in metrics_result.items():
    print(f"  {k:<35}: {v}")
print(f"\nSaved → {OUT_DIR}/metrics_full_tinyllama.json")

## Cell 6 — Ablation Study: 97 Questions  (§2.4)
Re-uses the already-loaded TinyLlama pipeline. Runs each question once, then re-weights signals offline.  
Expected time: **~15-20 min** (same pipeline calls as Cell 5, re-used if cached).

In [ ]:
import json
import statistics
from pathlib import Path

SIGNALS = [
    "retrieval",
    "generation",
    "consistency",
    "source_agreement",
    "entity_coverage",
]
SIGNAL_MAP = {
    "retrieval": "retrieval_confidence",
    "generation": "generation_confidence",
    "consistency": "consistency_score",
    "source_agreement": "source_agreement",
    "entity_coverage": "medical_entity_coverage",
}
ABLATION_VARIANTS = {
    "all_signals": {s: 1.0 for s in SIGNALS},
    "ablate_retrieval": {s: (0.0 if s == "retrieval" else 1.0) for s in SIGNALS},
    "ablate_generation": {s: (0.0 if s == "generation" else 1.0) for s in SIGNALS},
    "ablate_consistency": {s: (0.0 if s == "consistency" else 1.0) for s in SIGNALS},
    "ablate_source_agr": {
        s: (0.0 if s == "source_agreement" else 1.0) for s in SIGNALS
    },
    "ablate_entity_cov": {s: (0.0 if s == "entity_coverage" else 1.0) for s in SIGNALS},
}


def recompute_conf(breakdown, weights):
    total = sum(weights.values()) or 1.0
    w = {k: v / total for k, v in weights.items()}
    raw = sum(w.get(sig, 0) * breakdown.get(SIGNAL_MAP[sig], 0.0) for sig in SIGNALS)
    return float(np.clip(raw, 0.0, 1.0))


def compute_ece(confidences, labels, n_bins=10):
    ca, la = np.array(confidences), np.array(labels)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (ca >= lo) & (ca < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(ca)) * abs(la[mask].mean() - ca[mask].mean())
    return float(ece)


def bootstrap_ci(scores, n_bootstrap=1000, ci=0.95):
    arr = np.array(scores)
    boot = [
        np.mean(np.random.choice(arr, size=len(arr), replace=True))
        for _ in range(n_bootstrap)
    ]
    return (
        float(np.mean(arr)),
        float(np.percentile(boot, (1 - ci) / 2 * 100)),
        float(np.percentile(boot, (1 + ci) / 2 * 100)),
    )


# ── Step 1: Run all 97 questions once, collect signal breakdowns ──────────────
# (pipeline is already loaded from Cell 5)
print(f"Collecting signal breakdowns for {len(all_cases)} questions ...")
records = []
for i, case in enumerate(all_cases, 1):
    try:
        resp = pipeline.answer(case["query"], include_explanation=True)
        answer = resp.answer if hasattr(resp, "answer") else str(resp)
        bd = getattr(resp, "confidence_breakdown", None) or {}
        kws = case.get("expected_keywords", [])
        kw_cov = keyword_coverage(answer, kws)
        records.append(
            {
                "breakdown": bd,
                "kw_coverage": kw_cov,
                "kw_label": 1 if kw_cov >= 0.4 else 0,
            }
        )
        if i % 10 == 0 or i == len(all_cases):
            print(f"  {i}/{len(all_cases)} collected")
    except Exception as e:
        print(f"  [{i}] ERROR: {e}")

print(f"\nCollected {len(records)} records. Re-weighting offline ...")

# ── Step 2: Offline re-weighting for each ablation variant ───────────────────
ablation_results = []
for variant_name, weights in ABLATION_VARIANTS.items():
    confs = [recompute_conf(r["breakdown"], weights) for r in records]
    labels = [r["kw_label"] for r in records]
    kw_covs = [r["kw_coverage"] for r in records]
    ece = compute_ece(confs, labels)
    conf_mean, conf_lo, conf_hi = bootstrap_ci(confs)
    kw_mean_v, kw_lo_v, kw_hi_v = bootstrap_ci(kw_covs)
    ablation_results.append(
        {
            "variant": variant_name,
            "n": len(records),
            "mean_confidence": round(conf_mean, 4),
            "confidence_ci_95": [round(conf_lo, 4), round(conf_hi, 4)],
            "mean_kw_coverage": round(kw_mean_v, 4),
            "kw_coverage_ci_95": [round(kw_lo_v, 4), round(kw_hi_v, 4)],
            "ece": round(ece, 4),
        }
    )

OUT_DIR = Path(PROJECT_DIR) / "evaluation" / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "ablation_full.json", "w") as f:
    json.dump(ablation_results, f, indent=2)

print("\nAblation Study Results:")
print(f"{'Variant':<25} {'Confidence':>12} {'KW Coverage':>12} {'ECE':>8}")
print("-" * 62)
for r in ablation_results:
    print(
        f"{r['variant']:<25} {r['mean_confidence']:>12.4f} {r['mean_kw_coverage']:>12.4f} {r['ece']:>8.4f}"
    )
print(f"\nSaved → {OUT_DIR}/ablation_full.json")

## Cell 7 — BioMistral 7B Comparative Evaluation  (§4.1)
Requires `models/biomistral/ggml-model-Q4_K_M.gguf` to be present (copied from Drive in Cell 3).  
Expected time: **~30-45 min** on T4 (GGUF runs faster with CUDA).

In [ ]:
import os
import json
import time
import statistics
from pathlib import Path

GGUF_PATH = Path(PROJECT_DIR) / "models" / "biomistral" / "ggml-model-Q4_K_M.gguf"

if not GGUF_PATH.exists():
    print(f"ERROR: BioMistral GGUF not found at {GGUF_PATH}")
    print(
        "Upload it to Google Drive under healthcare_qa_data/models/biomistral/ and re-run Cell 3."
    )
else:
    print(
        f"Found BioMistral GGUF: {GGUF_PATH} ({GGUF_PATH.stat().st_size / 1e9:.2f} GB)"
    )

    # Load BioMistral pipeline
    print("Loading BioMistral pipeline ...")
    from api.main import get_pipeline

    pipeline_bio = get_pipeline("biomistral")
    print("BioMistral pipeline loaded.")

    # Run evaluation on all 97 questions
    from rouge_score import rouge_scorer as _rs
    import evaluate as ev

    scorer_rouge = _rs.RougeScorer(["rougeL"], use_stemmer=True)
    bertscore_metric = ev.load("bertscore")

    preds_bio, refs_bio = [], []
    kw_scores_bio, rouge_scores_bio, latencies_bio = [], [], []
    answerable_bio = 0

    print(f"\nRunning {len(all_cases)} questions with BioMistral ...")
    for i, case in enumerate(all_cases, 1):
        try:
            t0 = time.perf_counter()
            resp = pipeline_bio.answer(case["query"], include_explanation=False)
            elapsed_ms = (time.perf_counter() - t0) * 1000
            latencies_bio.append(elapsed_ms)

            answer = resp.answer if hasattr(resp, "answer") else str(resp)
            ref = case.get("reference_answer", "")
            kws = case.get("expected_keywords", [])

            kw_scores_bio.append(keyword_coverage(answer, kws))
            if ref:
                rouge_scores_bio.append(
                    scorer_rouge.score(ref, answer)["rougeL"].fmeasure
                )
                preds_bio.append(answer)
                refs_bio.append(ref)
            if getattr(resp, "is_answerable", True):
                answerable_bio += 1

            if i % 10 == 0 or i == len(all_cases):
                print(
                    f"  {i}/{len(all_cases)}  kw={statistics.mean(kw_scores_bio):.3f}  "
                    f"lat={statistics.mean(latencies_bio):.0f}ms"
                )
        except Exception as e:
            print(f"  [{i}] ERROR: {e}")

    # BERTScore
    bs_f1_bio = 0.0
    if preds_bio:
        try:
            bs_bio = bertscore_metric.compute(
                predictions=preds_bio,
                references=refs_bio,
                lang="en",
                rescale_with_baseline=True,
                device="cuda" if torch.cuda.is_available() else "cpu",
            )
        except:
            bs_bio = bertscore_metric.compute(
                predictions=preds_bio,
                references=refs_bio,
                lang="en",
                rescale_with_baseline=False,
                device="cuda" if torch.cuda.is_available() else "cpu",
            )
        bs_f1_bio = statistics.mean(bs_bio["f1"])

    kw_mean_b, kw_lo_b, kw_hi_b = bootstrap_ci(kw_scores_bio)
    r_mean_b, r_lo_b, r_hi_b = (
        bootstrap_ci(rouge_scores_bio) if rouge_scores_bio else (0, 0, 0)
    )
    bs_mean_b, bs_lo_b, bs_hi_b = bootstrap_ci(bs_bio["f1"]) if preds_bio else (0, 0, 0)

    metrics_bio = {
        "variant": "standard_biomistral",
        "n": len(kw_scores_bio),
        "answerable_pct": round(answerable_bio / max(len(kw_scores_bio), 1), 4),
        "keyword_coverage_mean": round(kw_mean_b, 4),
        "keyword_coverage_ci_95": [round(kw_lo_b, 4), round(kw_hi_b, 4)],
        "rougeL_mean": round(r_mean_b, 4),
        "rougeL_ci_95": [round(r_lo_b, 4), round(r_hi_b, 4)],
        "bertscore_f1_mean": round(bs_f1_bio, 4),
        "bertscore_ci_95": [round(bs_lo_b, 4), round(bs_hi_b, 4)],
        "latency_mean_ms": round(statistics.mean(latencies_bio), 1)
        if latencies_bio
        else 0,
        "latency_p90_ms": round(sorted(latencies_bio)[int(0.9 * len(latencies_bio))], 1)
        if latencies_bio
        else 0,
    }

    OUT_DIR = Path(PROJECT_DIR) / "evaluation" / "results"
    with open(OUT_DIR / "metrics_full_biomistral.json", "w") as f:
        json.dump(metrics_bio, f, indent=2)

    print("\nBioMistral Results:")
    for k, v in metrics_bio.items():
        print(f"  {k:<35}: {v}")
    print(f"\nSaved → {OUT_DIR}/metrics_full_biomistral.json")

## Cell 8 — QLoRA Re-Training: 300 Steps  (§4.2)
Downloads TinyLlama from HuggingFace and fine-tunes with QLoRA on the trajectory data.  
Expected time: **~10-15 min** on T4 GPU.

In [ ]:
import os
import json
import sys
from pathlib import Path
import torch

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# ── Load training data from trajectory log ───────────────────────────────────
TRAJ_PATH = Path(PROJECT_DIR) / "data" / "feedback" / "response_trajectories.jsonl"
training_records = []

if TRAJ_PATH.exists():
    with open(TRAJ_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    training_records.append(json.loads(line))
                except:
                    pass
    print(f"Loaded {len(training_records)} trajectory records.")
else:
    print(f"WARNING: Trajectory file not found at {TRAJ_PATH}")
    print("Using test set as fallback training data...")
    with open(Path(PROJECT_DIR) / "evaluation" / "test_set_v2.json") as f:
        ts = json.load(f)["test_cases"]
    training_records = [
        {"question": c["query"], "answer": c.get("reference_answer", "")} for c in ts
    ]
    print(f"Loaded {len(training_records)} test cases as training data.")

# ── Format as instruction-following examples ──────────────────────────────────
SYSTEM_PROMPT = (
    "You are a helpful medical assistant. Answer the following healthcare question "
    "accurately and safely. Always recommend consulting a healthcare professional "
    "for personal medical advice."
)


def format_example(record):
    question = record.get("question", record.get("query", ""))
    answer = record.get(
        "answer", record.get("response", record.get("reference_answer", ""))
    )
    if not question or not answer:
        return None
    return (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
        f"<|user|>\n{question}</s>\n"
        f"<|assistant|>\n{answer}</s>"
    )


texts = [t for r in training_records if (t := format_example(r)) is not None]
print(f"Formatted {len(texts)} training examples.")
print("Example:", texts[0][:200] if texts else "(none)")

# ── QLoRA Training ────────────────────────────────────────────────────────────
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = str(Path(PROJECT_DIR) / "models" / "fine_tuned_colab")

print(f"\nLoading {MODEL_NAME} in 4-bit ...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
print("Base model loaded.")

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# Tokenize
def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )


dataset = Dataset.from_dict({"text": texts})
tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

# Training args — 300 steps
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=300,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=25,
    save_steps=150,
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

print("\nStarting QLoRA training (300 steps) ...")
train_result = trainer.train()

# Save adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Log training info
train_log = {
    "base_model": MODEL_NAME,
    "steps": train_result.global_step,
    "train_loss": train_result.training_loss,
    "train_samples": len(texts),
    "lora_r": 16,
    "lora_alpha": 32,
    "output_dir": OUTPUT_DIR,
}
with open(
    Path(PROJECT_DIR) / "evaluation" / "results" / "qlora_training_log.json", "w"
) as f:
    json.dump(train_log, f, indent=2)

print(
    f"\nTraining complete! Steps={train_result.global_step}, Loss={train_result.training_loss:.4f}"
)
print(f"Adapter saved → {OUTPUT_DIR}")

## Cell 9 — QLoRA Model Evaluation  (§4.2 continued)
Evaluates the fine-tuned adapter on all 97 questions.  
Expected time: **~15-20 min** on T4.

In [ ]:
import json
import time
import statistics
from pathlib import Path
import torch
from rouge_score import rouge_scorer as _rs
import evaluate as ev

ADAPTER_DIR = Path(PROJECT_DIR) / "models" / "fine_tuned_colab"

if not ADAPTER_DIR.exists():
    # Fall back to the original adapter if colab training didn't run
    ADAPTER_DIR = Path(PROJECT_DIR) / "models" / "fine_tuned"
    print(f"Using pre-existing adapter: {ADAPTER_DIR}")
else:
    print(f"Using Colab-trained adapter: {ADAPTER_DIR}")

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading base model + LoRA adapter for inference ...")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer_ql = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model_ql = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_cfg,
    device_map="auto",
)
qlora_model = PeftModel.from_pretrained(base_model_ql, str(ADAPTER_DIR))
qlora_model.eval()
print("Model loaded.")

SYSTEM_PROMPT = (
    "You are a helpful medical assistant. Answer accurately and safely. "
    "Recommend consulting a healthcare professional for personal medical advice."
)


def qlora_generate(question, max_new_tokens=256):
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n<|user|>\n{question}</s>\n<|assistant|>\n"
    )
    inputs = tokenizer_ql(prompt, return_tensors="pt").to(qlora_model.device)
    with torch.no_grad():
        output = qlora_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer_ql.eos_token_id,
        )
    generated = tokenizer_ql.decode(
        output[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return generated.strip()


# Run evaluation
scorer_rouge_ql = _rs.RougeScorer(["rougeL"], use_stemmer=True)
bertscore_ql = ev.load("bertscore")

preds_ql, refs_ql = [], []
kw_scores_ql, rouge_scores_ql, latencies_ql = [], [], []

print(f"\nEvaluating QLoRA model on {len(all_cases)} questions ...")
for i, case in enumerate(all_cases, 1):
    try:
        t0 = time.perf_counter()
        answer = qlora_generate(case["query"])
        elapsed_ms = (time.perf_counter() - t0) * 1000
        latencies_ql.append(elapsed_ms)

        ref = case.get("reference_answer", "")
        kws = case.get("expected_keywords", [])
        kw_scores_ql.append(keyword_coverage(answer, kws))
        if ref:
            rouge_scores_ql.append(
                scorer_rouge_ql.score(ref, answer)["rougeL"].fmeasure
            )
            preds_ql.append(answer)
            refs_ql.append(ref)

        if i % 10 == 0 or i == len(all_cases):
            print(
                f"  {i}/{len(all_cases)}  kw={statistics.mean(kw_scores_ql):.3f}  "
                f"lat={statistics.mean(latencies_ql):.0f}ms"
            )
    except Exception as e:
        print(f"  [{i}] ERROR: {e}")

# BERTScore
bs_f1_ql = 0.0
if preds_ql:
    try:
        bs_ql = bertscore_ql.compute(
            predictions=preds_ql,
            references=refs_ql,
            lang="en",
            rescale_with_baseline=True,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
    except:
        bs_ql = bertscore_ql.compute(
            predictions=preds_ql,
            references=refs_ql,
            lang="en",
            rescale_with_baseline=False,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
    bs_f1_ql = statistics.mean(bs_ql["f1"])

kw_m_ql, kw_lo_ql, kw_hi_ql = bootstrap_ci(kw_scores_ql)
r_m_ql, r_lo_ql, r_hi_ql = (
    bootstrap_ci(rouge_scores_ql) if rouge_scores_ql else (0, 0, 0)
)
bs_m_ql, bs_lo_ql, bs_hi_ql = bootstrap_ci(bs_ql["f1"]) if preds_ql else (0, 0, 0)

metrics_qlora = {
    "variant": "qlora_finetuned_tinyllama",
    "n": len(kw_scores_ql),
    "keyword_coverage_mean": round(kw_m_ql, 4),
    "keyword_coverage_ci_95": [round(kw_lo_ql, 4), round(kw_hi_ql, 4)],
    "rougeL_mean": round(r_m_ql, 4),
    "rougeL_ci_95": [round(r_lo_ql, 4), round(r_hi_ql, 4)],
    "bertscore_f1_mean": round(bs_f1_ql, 4),
    "bertscore_ci_95": [round(bs_lo_ql, 4), round(bs_hi_ql, 4)],
    "latency_mean_ms": round(statistics.mean(latencies_ql), 1) if latencies_ql else 0,
    "latency_p90_ms": round(sorted(latencies_ql)[int(0.9 * len(latencies_ql))], 1)
    if latencies_ql
    else 0,
    "adapter_dir": str(ADAPTER_DIR),
}

OUT_DIR = Path(PROJECT_DIR) / "evaluation" / "results"
with open(OUT_DIR / "metrics_full_qlora.json", "w") as f:
    json.dump(metrics_qlora, f, indent=2)

print("\nQLoRA Results:")
for k, v in metrics_qlora.items():
    print(f"  {k:<35}: {v}")
print(f"\nSaved → {OUT_DIR}/metrics_full_qlora.json")

## Cell 10 — Baseline Comparisons  (§4.5)
Three baselines vs. the full pipeline:
- **No-RAG**: TinyLlama with no retrieval context
- **Dense-only**: Vector search only (no BM25, no RRF)
- **No-XAI**: Full retrieval but skip confidence scoring and hallucination detection

Expected time: **~30-45 min** total.

In [ ]:
import json
import time
import statistics
from pathlib import Path
import torch
from rouge_score import rouge_scorer as _rs
import evaluate as ev
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline as hf_pipeline

OUT_DIR = Path(PROJECT_DIR) / "evaluation" / "results"
scorer_rouge_bl = _rs.RougeScorer(["rougeL"], use_stemmer=True)
bertscore_bl = ev.load("bertscore")
device_str = "cuda" if torch.cuda.is_available() else "cpu"


def eval_answers(answers, cases):
    """Compute metrics given a list of answers and test cases."""
    kw_scores, rouge_scores, preds, refs = [], [], [], []
    for answer, case in zip(answers, cases):
        kws = case.get("expected_keywords", [])
        ref = case.get("reference_answer", "")
        kw_scores.append(keyword_coverage(answer, kws))
        if ref:
            rouge_scores.append(scorer_rouge_bl.score(ref, answer)["rougeL"].fmeasure)
            preds.append(answer)
            refs.append(ref)

    bs_f1 = 0.0
    if preds:
        try:
            bs = bertscore_bl.compute(
                predictions=preds,
                references=refs,
                lang="en",
                rescale_with_baseline=True,
                device=device_str,
            )
        except:
            bs = bertscore_bl.compute(
                predictions=preds,
                references=refs,
                lang="en",
                rescale_with_baseline=False,
                device=device_str,
            )
        bs_f1 = statistics.mean(bs["f1"])

    kw_m, kw_lo, kw_hi = bootstrap_ci(kw_scores)
    r_m, r_lo, r_hi = bootstrap_ci(rouge_scores) if rouge_scores else (0, 0, 0)
    return {
        "n": len(kw_scores),
        "keyword_coverage_mean": round(kw_m, 4),
        "keyword_coverage_ci_95": [round(kw_lo, 4), round(kw_hi, 4)],
        "rougeL_mean": round(r_m, 4),
        "rougeL_ci_95": [round(r_lo, 4), round(r_hi, 4)],
        "bertscore_f1_mean": round(bs_f1, 4),
    }


# ── Load shared base LLM (TinyLlama) ────────────────────────────────────────
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print("Loading base TinyLlama for baseline experiments ...")
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_lm = hf_pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=base_tokenizer,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=256,
)
print("TinyLlama loaded.")

SYSTEM_PROMPT = (
    "You are a helpful medical assistant. Answer the question accurately and safely."
)


def no_rag_answer(question):
    """Baseline 1: No RAG — just LLM."""
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n<|user|>\n{question}</s>\n<|assistant|>\n"
    )
    out = base_lm(prompt, return_full_text=False)
    return out[0]["generated_text"].strip()


# ── Baseline 1: No-RAG ────────────────────────────────────────────────────────
print(f"\n--- Baseline 1: No-RAG ({len(all_cases)} questions) ---")
answers_norag = []
latencies_norag = []
for i, case in enumerate(all_cases, 1):
    try:
        t0 = time.perf_counter()
        ans = no_rag_answer(case["query"])
        latencies_norag.append((time.perf_counter() - t0) * 1000)
        answers_norag.append(ans)
    except Exception as e:
        answers_norag.append("")
        print(f"  [{i}] ERROR: {e}")
    if i % 10 == 0 or i == len(all_cases):
        print(f"  {i}/{len(all_cases)} done")

metrics_norag = eval_answers(answers_norag, all_cases)
metrics_norag["variant"] = "baseline_no_rag"
metrics_norag["latency_mean_ms"] = (
    round(statistics.mean(latencies_norag), 1) if latencies_norag else 0
)
print(
    f"  KW coverage: {metrics_norag['keyword_coverage_mean']:.4f}  ROUGE-L: {metrics_norag['rougeL_mean']:.4f}"
)

# ── Baseline 2: Dense-Only Retrieval ─────────────────────────────────────────
print(f"\n--- Baseline 2: Dense-Only ({len(all_cases)} questions) ---")

from config.settings import Config

cfg_dense = Config()
cfg_dense.retrieval.dense_weight = 1.0
cfg_dense.retrieval.sparse_weight = 0.0
cfg_dense.pipeline.enable_query_enhancement = False
cfg_dense.pipeline.enable_reranker = False

from src.pipeline.qa_pipeline import HealthcareQAPipeline

pipeline_dense = HealthcareQAPipeline(cfg_dense)

answers_dense, latencies_dense = [], []
for i, case in enumerate(all_cases, 1):
    try:
        t0 = time.perf_counter()
        resp = pipeline_dense.answer(case["query"], include_explanation=False)
        latencies_dense.append((time.perf_counter() - t0) * 1000)
        answers_dense.append(resp.answer if hasattr(resp, "answer") else str(resp))
    except Exception as e:
        answers_dense.append("")
        print(f"  [{i}] ERROR: {e}")
    if i % 10 == 0 or i == len(all_cases):
        print(f"  {i}/{len(all_cases)} done")

metrics_dense = eval_answers(answers_dense, all_cases)
metrics_dense["variant"] = "baseline_dense_only"
metrics_dense["latency_mean_ms"] = (
    round(statistics.mean(latencies_dense), 1) if latencies_dense else 0
)
print(
    f"  KW coverage: {metrics_dense['keyword_coverage_mean']:.4f}  ROUGE-L: {metrics_dense['rougeL_mean']:.4f}"
)

# ── Baseline 3: No-XAI Pipeline ──────────────────────────────────────────────
print(f"\n--- Baseline 3: No-XAI ({len(all_cases)} questions) ---")

cfg_noxai = Config()
cfg_noxai.pipeline.enable_factual_consistency = False
cfg_noxai.pipeline.enable_corrective_rag = False

pipeline_noxai = HealthcareQAPipeline(cfg_noxai)

answers_noxai, latencies_noxai = [], []
for i, case in enumerate(all_cases, 1):
    try:
        t0 = time.perf_counter()
        resp = pipeline_noxai.answer(case["query"], include_explanation=False)
        latencies_noxai.append((time.perf_counter() - t0) * 1000)
        answers_noxai.append(resp.answer if hasattr(resp, "answer") else str(resp))
    except Exception as e:
        answers_noxai.append("")
        print(f"  [{i}] ERROR: {e}")
    if i % 10 == 0 or i == len(all_cases):
        print(f"  {i}/{len(all_cases)} done")

metrics_noxai = eval_answers(answers_noxai, all_cases)
metrics_noxai["variant"] = "baseline_no_xai"
metrics_noxai["latency_mean_ms"] = (
    round(statistics.mean(latencies_noxai), 1) if latencies_noxai else 0
)
print(
    f"  KW coverage: {metrics_noxai['keyword_coverage_mean']:.4f}  ROUGE-L: {metrics_noxai['rougeL_mean']:.4f}"
)

# ── Save all baseline results ─────────────────────────────────────────────────
all_baselines = [metrics_norag, metrics_dense, metrics_noxai]
with open(OUT_DIR / "baselines.json", "w") as f:
    json.dump(all_baselines, f, indent=2)

print("\nBaseline Summary:")
print(f"{'Variant':<30} {'KW Coverage':>12} {'ROUGE-L':>10} {'BERTScore F1':>14}")
print("-" * 70)
for m in all_baselines:
    print(
        f"{m['variant']:<30} {m['keyword_coverage_mean']:>12.4f} {m['rougeL_mean']:>10.4f} {m['bertscore_f1_mean']:>14.4f}"
    )
print(f"\nSaved → {OUT_DIR}/baselines.json")

## Cell 11 — Generate Paper-Quality Figures  (§4.3)
Generates 6 publication-ready figures at 300 DPI (PNG + PDF).  
Loads results from Cells 5–10.

In [ ]:
import json
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "font.size": 12,
        "font.family": "serif",
        "figure.figsize": (8, 5),
        "figure.dpi": 300,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

COLORS = ["#4f46e5", "#e53e3e", "#38a169", "#d69e2e", "#805ad5"]
OUT_DIR = Path(PROJECT_DIR) / "evaluation" / "results" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def savefig(fig, name):
    fig.savefig(OUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {name}.png / .pdf")


# ── Load result files ─────────────────────────────────────────────────────────
RESULTS = Path(PROJECT_DIR) / "evaluation" / "results"


def load_json(path, default=None):
    try:
        with open(path) as f:
            return json.load(f)
    except:
        print(f"  WARNING: {path} not found, using default.")
        return default


metrics_tl = load_json(RESULTS / "metrics_full_tinyllama.json", {})
metrics_bm = load_json(RESULTS / "metrics_full_biomistral.json", {})
metrics_ql = load_json(RESULTS / "metrics_full_qlora.json", {})
ablation = load_json(RESULTS / "ablation_full.json", [])
calibration = load_json(RESULTS / "calibration.json", {})
latency_raw = load_json(RESULTS / "latency.json", [])
baselines = load_json(RESULTS / "baselines.json", [])

print("Results loaded.")

# ── Fig 1: System Architecture Diagram ───────────────────────────────────────
print("\nFig 1: Architecture diagram ...")
fig, ax = plt.subplots(figsize=(12, 3))
ax.set_xlim(0, 12)
ax.set_ylim(0, 1)
ax.axis("off")

stages = [
    ("User\nQuestion", COLORS[0]),
    ("Safety\nCheck", COLORS[1]),
    ("Query\nEnhancer", COLORS[2]),
    ("Hybrid\nRetrieval", COLORS[0]),
    ("Corrective\nRAG", COLORS[2]),
    ("LLM\nGeneration", COLORS[0]),
    ("Hallucination\nDetect", COLORS[1]),
    ("Confidence\nScoring", COLORS[3]),
    ("Answer +\nXAI", COLORS[4]),
]
box_w, box_h = 1.1, 0.55
y_center = 0.5
start_x = 0.5
gap = (12 - start_x * 2 - box_w * len(stages)) / max(len(stages) - 1, 1)

for idx, (label, color) in enumerate(stages):
    x = start_x + idx * (box_w + gap)
    rect = mpatches.FancyBboxPatch(
        (x - box_w / 2, y_center - box_h / 2),
        box_w,
        box_h,
        boxstyle="round,pad=0.05",
        linewidth=1,
        edgecolor="black",
        facecolor=color,
        alpha=0.75,
    )
    ax.add_patch(rect)
    ax.text(
        x,
        y_center,
        label,
        ha="center",
        va="center",
        fontsize=7,
        color="white",
        fontweight="bold",
        wrap=True,
    )
    if idx < len(stages) - 1:
        next_x = start_x + (idx + 1) * (box_w + gap)
        ax.annotate(
            "",
            xy=(next_x - box_w / 2 - 0.02, y_center),
            xytext=(x + box_w / 2 + 0.02, y_center),
            arrowprops=dict(arrowstyle="->", color="#333", lw=1.2),
        )

ax.set_title("Healthcare QA Chatbot — Pipeline Architecture", fontsize=13, pad=8)
savefig(fig, "fig1_architecture")

# ── Fig 2: Model Comparison Bar Chart ────────────────────────────────────────
print("Fig 2: Model comparison ...")
models_data = []
for name, data in [
    ("TinyLlama\n1.1B", metrics_tl),
    ("BioMistral\n7B Q4", metrics_bm),
    ("QLoRA\nFine-tuned", metrics_ql),
]:
    if data:
        models_data.append(
            {
                "name": name,
                "kw": data.get("keyword_coverage_mean", 0),
                "kw_err": (
                    data.get("keyword_coverage_ci_95", [0, 0])[1]
                    - data.get("keyword_coverage_ci_95", [0, 0])[0]
                )
                / 2,
                "rouge": data.get("rougeL_mean", 0),
                "rouge_err": (
                    data.get("rougeL_ci_95", [0, 0])[1]
                    - data.get("rougeL_ci_95", [0, 0])[0]
                )
                / 2,
                "bert": data.get("bertscore_f1_mean", 0),
                "bert_err": (
                    data.get("bertscore_ci_95", [0, 0])[1]
                    - data.get("bertscore_ci_95", [0, 0])[0]
                )
                / 2,
            }
        )

if models_data:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(models_data))
    w = 0.25
    metrics_labels = [
        ("KW Coverage", "kw", "kw_err"),
        ("ROUGE-L", "rouge", "rouge_err"),
        ("BERTScore F1", "bert", "bert_err"),
    ]
    for i, (label, key, err_key) in enumerate(metrics_labels):
        vals = [d[key] for d in models_data]
        errs = [d[err_key] for d in models_data]
        bars = ax.bar(
            x + (i - 1) * w,
            vals,
            w,
            label=label,
            color=COLORS[i],
            alpha=0.85,
            yerr=errs,
            capsize=4,
            edgecolor="black",
            linewidth=0.5,
        )
    ax.set_xticks(x)
    ax.set_xticklabels([d["name"] for d in models_data])
    ax.set_ylabel("Score")
    ax.set_title("Model Comparison — Quality Metrics (97 Questions, 95% CI)")
    ax.legend()
    ax.set_ylim(0, 1)
    savefig(fig, "fig2_model_comparison")
else:
    print("  Skipping Fig 2 (no model data available).")

# ── Fig 3: Reliability Diagram (Before/After Calibration) ────────────────────
print("Fig 3: Reliability diagram ...")
if calibration and "bin_confidences" in calibration:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration")
    bin_confs = calibration.get("bin_confidences", [])
    bin_accs = calibration.get("bin_accuracies", [])
    if bin_confs and bin_accs:
        ax.bar(
            bin_confs,
            bin_accs,
            width=0.08,
            alpha=0.7,
            color=COLORS[0],
            edgecolor="black",
            label="Model (calibrated)",
        )
    ece = calibration.get("ece", 0)
    ax.set_xlabel("Mean Predicted Confidence")
    ax.set_ylabel("Fraction Correct (KW coverage >= 0.4)")
    ax.set_title(f"Reliability Diagram  (ECE = {ece:.3f})")
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    savefig(fig, "fig3_reliability_diagram")
else:
    print("  Skipping Fig 3 (calibration data not available).")

# ── Fig 4: Ablation Study Grouped Bar Chart ───────────────────────────────────
print("Fig 4: Ablation study ...")
if ablation:
    labels_x = [
        r["variant"].replace("ablate_", "-").replace("_", " ") for r in ablation
    ]
    x = np.arange(len(labels_x))
    conf_vals = [r.get("mean_confidence", 0) for r in ablation]
    ece_vals = [r.get("ece", 0) for r in ablation]
    kw_vals = [r.get("mean_kw_coverage", 0) for r in ablation]
    conf_errs = [
        (r.get("confidence_ci_95", [v, v])[1] - r.get("confidence_ci_95", [v, v])[0])
        / 2
        for r, v in zip(ablation, conf_vals)
    ]
    kw_errs = [
        (r.get("kw_coverage_ci_95", [v, v])[1] - r.get("kw_coverage_ci_95", [v, v])[0])
        / 2
        for r, v in zip(ablation, kw_vals)
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax_sub, vals, errs, title, color in zip(
        axes,
        [conf_vals, ece_vals, kw_vals],
        [conf_errs, [0] * len(ece_vals), kw_errs],
        ["Mean Confidence", "ECE (↓ better)", "Mean KW Coverage"],
        [COLORS[0], COLORS[1], COLORS[2]],
    ):
        bars = ax_sub.bar(
            x,
            vals,
            color=color,
            alpha=0.82,
            edgecolor="black",
            linewidth=0.8,
            yerr=errs,
            capsize=5,
        )
        ax_sub.set_xticks(x)
        ax_sub.set_xticklabels(labels_x, rotation=35, ha="right", fontsize=9)
        ax_sub.set_title(title, fontsize=11)
        ax_sub.set_ylim(0, max(vals) * 1.25 if vals else 1)
        bars[0].set_edgecolor("gold")
        bars[0].set_linewidth(2.5)
        for bar, v in zip(bars, vals):
            ax_sub.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.004,
                f"{v:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )

    fig.suptitle(
        f"Confidence Signal Ablation (n={ablation[0].get('n', '?')} questions, 95% CI)",
        fontsize=13,
    )
    fig.tight_layout()
    savefig(fig, "fig4_ablation")
else:
    print("  Skipping Fig 4 (ablation data not available).")

# ── Fig 5: Latency Breakdown Stacked Bar ─────────────────────────────────────
print("Fig 5: Latency breakdown ...")
if latency_raw:
    row = latency_raw[0] if isinstance(latency_raw, list) else latency_raw
    stage_keys = [k for k in row if k.endswith("_mean") and k != "total_ms_mean"]
    stage_names = [k.replace("_mean", "").replace("_", " ") for k in stage_keys]
    stage_vals = [row[k] for k in stage_keys]

    fig, ax = plt.subplots(figsize=(9, 5))
    bottom = 0
    for name, val, color in zip(stage_names, stage_vals, COLORS * 5):
        ax.bar(
            0,
            val,
            bottom=bottom,
            color=color,
            alpha=0.85,
            edgecolor="black",
            linewidth=0.5,
            label=f"{name} ({val:.0f}ms)",
        )
        if val > 50:
            ax.text(
                0,
                bottom + val / 2,
                f"{name}\n{val:.0f}ms",
                ha="center",
                va="center",
                fontsize=9,
                color="white",
                fontweight="bold",
            )
        bottom += val

    total = row.get("total_ms_mean", bottom)
    ax.set_xticks([0])
    ax.set_xticklabels(["Standard Pipeline"])
    ax.set_ylabel("Latency (ms)")
    ax.set_title(f"Pipeline Stage Latency Breakdown (Total ≈ {total:.0f}ms)")
    ax.legend(loc="upper right", fontsize=9)
    savefig(fig, "fig5_latency_breakdown")
else:
    print("  Skipping Fig 5 (latency data not available).")

# ── Fig 6: Baseline Comparison ────────────────────────────────────────────────
print("Fig 6: Baseline comparison ...")
if baselines and metrics_tl:
    all_models_cmp = [
        {
            "name": "Full\nPipeline",
            "kw": metrics_tl.get("keyword_coverage_mean", 0),
            "rouge": metrics_tl.get("rougeL_mean", 0),
            "bert": metrics_tl.get("bertscore_f1_mean", 0),
        },
    ]
    label_map = {
        "baseline_no_rag": "No RAG",
        "baseline_dense_only": "Dense Only",
        "baseline_no_xai": "No XAI",
    }
    for bl in baselines:
        all_models_cmp.append(
            {
                "name": label_map.get(bl["variant"], bl["variant"]),
                "kw": bl.get("keyword_coverage_mean", 0),
                "rouge": bl.get("rougeL_mean", 0),
                "bert": bl.get("bertscore_f1_mean", 0),
            }
        )

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(all_models_cmp))
    w = 0.25
    for i, (label, key) in enumerate(
        [("KW Coverage", "kw"), ("ROUGE-L", "rouge"), ("BERTScore F1", "bert")]
    ):
        vals = [d[key] for d in all_models_cmp]
        bars = ax.bar(
            x + (i - 1) * w,
            vals,
            w,
            label=label,
            color=COLORS[i],
            alpha=0.85,
            edgecolor="black",
            linewidth=0.5,
        )
        bars[0].set_edgecolor("gold")
        bars[0].set_linewidth(2)

    ax.set_xticks(x)
    ax.set_xticklabels([d["name"] for d in all_models_cmp])
    ax.set_ylabel("Score")
    ax.set_title("Full Pipeline vs Baselines (97 Questions)")
    ax.legend()
    ax.set_ylim(0, 1)
    ax.axvline(0.5, color="gold", lw=1, ls="--", alpha=0.5)
    savefig(fig, "fig6_baseline_comparison")
else:
    print("  Skipping Fig 6 (baseline or full-pipeline data not available).")

print(f"\nAll figures saved to {OUT_DIR}")

## Cell 12 — Save Results Back to Google Drive & GitHub
Run this after all evaluation cells complete.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

RESULTS_DIR = Path(PROJECT_DIR) / "evaluation" / "results"
DRIVE_RESULTS = f"{DRIVE_BASE}/results_colab"

# ── Copy to Google Drive ─────────────────────────────────────────────────────
print("Copying results to Google Drive ...")
os.makedirs(DRIVE_RESULTS, exist_ok=True)

for f in RESULTS_DIR.rglob("*"):
    if f.is_file():
        rel = f.relative_to(RESULTS_DIR)
        dst = Path(DRIVE_RESULTS) / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)

print(f"Results copied to {DRIVE_RESULTS}")
print("Files saved:")
for f in sorted(Path(DRIVE_RESULTS).rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(DRIVE_RESULTS)}")

# ── Optional: Push to GitHub ──────────────────────────────────────────────────
if GITHUB_TOKEN:
    print("\nPushing results to GitHub ...")
    repo_url_with_token = GITHUB_REPO.replace("https://", f"https://{GITHUB_TOKEN}@")
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "remote", "set-url", "origin", repo_url_with_token]
    )
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "config", "user.email", "colab@runner.local"]
    )
    subprocess.run(["git", "-C", PROJECT_DIR, "config", "user.name", "Colab Runner"])
    subprocess.run(["git", "-C", PROJECT_DIR, "add", "evaluation/results/"])
    subprocess.run(
        [
            "git",
            "-C",
            PROJECT_DIR,
            "commit",
            "-m",
            "chore: add Colab evaluation results (97-question full run, all models)",
        ]
    )
    result = subprocess.run(
        ["git", "-C", PROJECT_DIR, "push"], capture_output=True, text=True
    )
    if result.returncode == 0:
        print("Pushed to GitHub successfully.")
    else:
        print(f"Push failed: {result.stderr}")
else:
    print("\nGITHUB_TOKEN not set — skipping GitHub push.")
    print("To push manually, run in a code cell:")
    print(
        "  !cd /content/project && git add evaluation/results/ && git commit -m 'add colab results' && git push"
    )

print("\nAll done!")

## Summary of Output Files

After running all cells, the following files are produced in `evaluation/results/`:

| File | Contents | Used in paper |
|------|----------|---------------|
| `metrics_full_tinyllama.json` | 97-Q metrics + 95% CI (TinyLlama) | Table 1 |
| `metrics_full_biomistral.json` | 97-Q metrics + 95% CI (BioMistral) | Table 1 |
| `metrics_full_qlora.json` | 97-Q metrics + 95% CI (QLoRA) | Table 1 |
| `ablation_full.json` | Ablation study, 97 questions, all variants | Table 2 / Fig 4 |
| `qlora_training_log.json` | QLoRA training metadata | §4.2 |
| `baselines.json` | No-RAG / Dense-only / No-XAI baselines | Table 3 |
| `figures/fig1_architecture.png` | Pipeline architecture diagram | Fig 1 |
| `figures/fig2_model_comparison.png` | Model quality bar chart | Fig 2 |
| `figures/fig3_reliability_diagram.png` | Calibration reliability plot | Fig 3 |
| `figures/fig4_ablation.png` | Ablation grouped bar chart | Fig 4 |
| `figures/fig5_latency_breakdown.png` | Latency stacked bar | Fig 5 |
| `figures/fig6_baseline_comparison.png` | Full pipeline vs baselines | Fig 6 |